In [201]:
import pandas as pd
import numpy as np
import sqlite3

In [202]:
fights_df = pd.read_csv('processed_fight_dat.csv')

conn = sqlite3.connect('fights.db')

fights_df.to_sql('fights', conn, if_exists='replace', index=True)

8604

In [203]:
print(fights_df.shape)
fights_df.head()

(8604, 310)


,Unnamed: 0,matchup,method,round,time,time format,referee,details:,fighters_a,fighters_b,...,Round 5 landed_clinch_a,Round 5 attempted_clinch_a,Round 5 landed_clinch_b,Round 5 attempted_clinch_b,Round 5 landed_ground_a,Round 5 attempted_ground_a,Round 5 landed_ground_b,Round 5 attempted_ground_b,event_date,event_location
0,0,Marlon Vera vs Dominick Cruz,KO/TKO,4,2:17,5 Rnd (5-5-5-5-5),Herb Dean,Kick to Head At Distance,Marlon Vera,Dominick Cruz,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
1,1,Nate Landwehr vs David Onama,Decision - Majority,3,5:00,3 Rnd (5-5-5),Mike Beltran,Mike Bell 28 - 28.Junichiro Kamijo 27 - 29.Der...,Nate Landwehr,David Onama,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
2,2,Yazmin Jauregui vs Iasmin Lucindo,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),Jason Herzog,Ron McCarthy 27 - 30.Derek Cleary 28 - 29.Sal ...,Yazmin Jauregui,Iasmin Lucindo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
3,3,Devin Clark vs Azamat Murzakanov,KO/TKO,3,1:18,3 Rnd (5-5-5),Frank Trigg,Punch to Body At Distance,Devin Clark,Azamat Murzakanov,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
4,4,Priscila Cachoeira vs Ariane da Silva,KO/TKO,1,1:05,3 Rnd (5-5-5),Herb Dean,Punches to Head At Distance,Priscila Cachoeira,Ariane da Silva,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"


In [204]:
# fights_df['event_date']
fights_df.drop(columns=['Unnamed: 0'], inplace = True)
fights_df['fight_id'] = fights_df.index + 1

fights_df.columns.tolist()

['matchup',
 'method',
 'round',
 'time',
 'time format',
 'referee',
 'details:',
 'fighters_a',
 'fighters_b',
 'knockdowns_a',
 'knockdowns_b',
 'landed_significant strikes_a',
 'attempted_significant strikes_a',
 'landed_significant strikes_b',
 'attempted_significant strikes_b',
 'significant strike percentage_a',
 'significant strike percentage_b',
 'landed_total strike_a',
 'attempted_total strike_a',
 'landed_total strike_b',
 'attempted_total strike_b',
 'landed_takedowns_a',
 'attempted_takedowns_a',
 'landed_takedowns_b',
 'attempted_takedowns_b',
 'takedown percentage_a',
 'takedown percentage_b',
 'submission attempts_a',
 'submission attempts_b',
 'reversals_a',
 'reversals_b',
 'control_a',
 'control_b',
 'Round 1 fighters_a',
 'Round 1 fighters_b',
 'Round 1 knockdowns_a',
 'Round 1 knockdowns_b',
 'Round 1 landed_significant strikes_a',
 'Round 1 attempted_significant strikes_a',
 'Round 1 landed_significant strikes_b',
 'Round 1 attempted_significant strikes_b',
 'Rou

## What I want to create

a fighter-fight table

#### Columns:
 - fighter (primary key)
 - fighter-fight-id 
 - all fight specific stats for the given fight
 - aggregate stats for the fight-specifc rows for the last 3 fights (will expand later)
 - aggregate stats for the fight-specific rows for the last 2 years of fights (will expand later)



### Pseudocode for the fighter-fight table creation
1) seperate columns into *_a, *_b, and universal 
2) combine universal to *_a, *_b columns
3) union the *_a and *_b columns
4) Deduplicate? there should be no duplicates
5) sort by fight date and set up an increasing numerical ID


### Pseudocode for the aggregate stats function
get_agg_stats(fighter, stat(s), time_period(s)):

1) get a subset of the total data that pertains to a specific fighter
2) for a given time period: get the selection of stats and return the min/max/average of each statistic

In [206]:
## seperate columns into *_a, *_b, and universal 
a_cols = [col for col in fights_df.columns.tolist() if col[-2:] == '_a']
b_cols = [col for col in fights_df.columns.tolist() if col[-2:] == '_b']
universal_cols = [col for col in fights_df.columns.tolist() if col[-2:] not in ['_a','_b']]

print(len(fights_df.columns) == len(a_cols) + len(b_cols) + len(universal_cols))

## combine universal to *_a, *_b columns
fighter_a_stats = fights_df[universal_cols + a_cols]
fighter_b_stats = fights_df[universal_cols + b_cols]

print(fighter_a_stats.shape)
print(fighter_b_stats.shape)

## rename the columns to no longer include _a or _b
a_cols_rename_dict = {col:col[:-2] for col in a_cols}
fighter_a_stats.rename(columns=a_cols_rename_dict, inplace=True)

b_cols_rename_dict = {col:col[:-2] for col in b_cols}
fighter_b_stats.rename(columns=b_cols_rename_dict, inplace=True)

## union the stat columns
fighter_stats = pd.concat([fighter_a_stats, fighter_b_stats], ignore_index = True)
print(fighter_stats.shape)

## Deduplicate? there should be no duplicates
fighter_stats_dedup = fighter_stats.drop_duplicates()
print('no duplicates:', fighter_stats.shape == fighter_stats_dedup.shape)

## sort by fight date and set up an increasing numerical ID
fighter_stats = fighter_stats.sort_values('event_date').reset_index(drop=True)
fighter_stats['fighter_fight_id'] = fighter_stats.index + 1

fighter_stats['event_date'] = pd.to_datetime(fighter_stats['event_date'])

fighter_stats.tail()


True
(8604, 160)
(8604, 160)
(17208, 160)
no duplicates: True


/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_35998/986011233.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fighter_a_stats.rename(columns=a_cols_rename_dict, inplace=True)
/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_35998/986011233.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fighter_b_stats.rename(columns=b_cols_rename_dict, inplace=True)


,matchup,method,round,time,time format,referee,details:,event_date,event_location,fight_id,...,Round 5 attempted_body,Round 5 landed_leg,Round 5 attempted_leg,Round 5 landed_distance,Round 5 attempted_distance,Round 5 landed_clinch,Round 5 attempted_clinch,Round 5 landed_ground,Round 5 attempted_ground,fighter_fight_id
17203,Tom Watson vs Brad Tavares,Decision - Split,3,5:00,3 Rnd (5-5-5),Leon Roberts,Howard Hughes 27 - 30.Doug Crosby 29 - 28.Chri...,2012-09-29,"Nottingham, England, United Kingdom",1403,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17204
17204,DaMarques Johnson vs Gunnar Nelson,Submission,1,3:34,3 Rnd (5-5-5),Herb Dean,Rear Naked Choke,2012-09-29,"Nottingham, England, United Kingdom",1404,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17205
17205,Jason Young vs Robert Peralta,KO/TKO,1,0:23,3 Rnd (5-5-5),Marc Goddard,Punch to Head At Distance,2012-09-29,"Nottingham, England, United Kingdom",1405,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17206
17206,Tom Watson vs Brad Tavares,Decision - Split,3,5:00,3 Rnd (5-5-5),Leon Roberts,Howard Hughes 27 - 30.Doug Crosby 29 - 28.Chri...,2012-09-29,"Nottingham, England, United Kingdom",1403,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17207
17207,Brad Pickett vs Yves Jabouin,KO/TKO,1,3:40,3 Rnd (5-5-5),Leon Roberts,Punch to Head At Distance,2012-09-29,"Nottingham, England, United Kingdom",1397,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17208


In [207]:
# ### Pseudocode for the aggregate stats function

# def get_agg_stats_recent_fights(fighter_stats, fighter, time_periods_fights, stats):

#     ## get a subset of the total data that pertains to a specific fighter
#     fighter_subset = fighter_stats[fighter_stats['fighters'] == fighter]

#     ## for a given time period: get the selection of stats and return the min/max/average of each statistic
#     for time_period in time_periods_fights:

#         fight_agg_stat = []

#         for stat in stats:

#             agg_col = stat + '_' + str(time_period)

#             fighter_stats[agg_col] = fighter_stats[stat].transform(lambda x: np.mean(x.tail(3)))


#     return fighter_stats


# test_agg = get_agg_stats_recent_fights(fighter_stats,'Stipe Miocic',[3],['attempted_significant strikes'])



In [208]:
## Upload fighter table into sqlite and try window functions

fighter_stats.to_sql('fighter_fights', conn, if_exists='replace', index=True)
fighter_stats

,matchup,method,round,time,time format,referee,details:,event_date,event_location,fight_id,...,Round 5 attempted_body,Round 5 landed_leg,Round 5 attempted_leg,Round 5 landed_distance,Round 5 attempted_distance,Round 5 landed_clinch,Round 5 attempted_clinch,Round 5 landed_ground,Round 5 attempted_ground,fighter_fight_id
0,Martin Kampmann vs Carlos Condit,Decision - Split,3,5:00,3 Rnd (5-5-5),Herb Dean,Doug Crosby 28 - 29.Cecil Peoples 29 - 28.Nels...,2009-04-01,"Nashville, Tennessee, USA",2613,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,Rob Kimmons vs Joe Vedepo,Submission,1,1:54,3 Rnd (5-5-5),Herb Dean,Guillotine Choke StandingTechnical Submission,2009-04-01,"Nashville, Tennessee, USA",2622,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,Aaron Simpson vs Tim McKenzie,KO/TKO,1,1:40,3 Rnd (5-5-5),Mario Yamasaki,Punches to Head On Ground,2009-04-01,"Nashville, Tennessee, USA",2623,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
3,Ricardo Almeida vs Matt Horwich,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),Dan Miragliotta,27 - 30.27 - 30.27 - 30.,2009-04-01,"Nashville, Tennessee, USA",2618,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
4,Brock Larson vs Jesse Sanders,Submission,1,2:01,3 Rnd (5-5-5),Herb Dean,Rear Naked Choke,2009-04-01,"Nashville, Tennessee, USA",2619,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17203,Tom Watson vs Brad Tavares,Decision - Split,3,5:00,3 Rnd (5-5-5),Leon Roberts,Howard Hughes 27 - 30.Doug Crosby 29 - 28.Chri...,2012-09-29,"Nottingham, England, United Kingdom",1403,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17204
17204,DaMarques Johnson vs Gunnar Nelson,Submission,1,3:34,3 Rnd (5-5-5),Herb Dean,Rear Naked Choke,2012-09-29,"Nottingham, England, United Kingdom",1404,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17205
17205,Jason Young vs Robert Peralta,KO/TKO,1,0:23,3 Rnd (5-5-5),Marc Goddard,Punch to Head At Distance,2012-09-29,"Nottingham, England, United Kingdom",1405,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17206
17206,Tom Watson vs Brad Tavares,Decision - Split,3,5:00,3 Rnd (5-5-5),Leon Roberts,Howard Hughes 27 - 30.Doug Crosby 29 - 28.Chri...,2012-09-29,"Nottingham, England, United Kingdom",1403,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17207


In [209]:
cur = conn.cursor()
cur.execute('select fighters, event_date, fighter_fight_id \
            from fighter_fights \
            where fighters = "Daniel Cormier" \
            order by date(event_date) DESC')

rows = cur.fetchall()
rows

[('Daniel Cormier', '2020-08-15 00:00:00', 2041),
 ('Daniel Cormier', '2019-08-17 00:00:00', 2142),
 ('Daniel Cormier', '2018-11-03 00:00:00', 12953),
 ('Daniel Cormier', '2018-07-07 00:00:00', 7165),
 ('Daniel Cormier', '2018-01-20 00:00:00', 6362),
 ('Daniel Cormier', '2017-07-29 00:00:00', 8360),
 ('Daniel Cormier', '2017-04-08 00:00:00', 289),
 ('Daniel Cormier', '2016-07-09 00:00:00', 7281),
 ('Daniel Cormier', '2015-10-03 00:00:00', 14541),
 ('Daniel Cormier', '2015-05-23 00:00:00', 12468),
 ('Daniel Cormier', '2015-01-03 00:00:00', 5842),
 ('Daniel Cormier', '2014-05-24 00:00:00', 12507),
 ('Daniel Cormier', '2014-02-22 00:00:00', 5341),
 ('Daniel Cormier', '2013-10-19 00:00:00', 15254),
 ('Daniel Cormier', '2013-04-20 00:00:00', 932)]

In [210]:
test_query = cur.execute('select * from fighter_fights where fighters = "Stipe Miocic"').fetchall()
len(test_query)

19

In [211]:
test_query_partition = cur.execute('select fighters, \
                                    event_date, \
                                    "attempted_significant strikes", \
                                    AVG("attempted_significant strikes") OVER(PARTITION BY fighters \
                                        ORDER BY event_date ASC \
                                        ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) AS Avg_sig_str_3_before \
                                    from fighter_fights \
                                    where fighters in ("Stipe Miocic", "Daniel Cormier")').fetchall()

print(len(test_query_partition))
test_query_partition


34


[('Daniel Cormier', '2013-04-20 00:00:00', 96, None),
 ('Daniel Cormier', '2013-10-19 00:00:00', 148, 96.0),
 ('Daniel Cormier', '2014-02-22 00:00:00', 37, 122.0),
 ('Daniel Cormier', '2014-05-24 00:00:00', 80, 93.66666666666667),
 ('Daniel Cormier', '2015-01-03 00:00:00', 165, 88.33333333333333),
 ('Daniel Cormier', '2015-05-23 00:00:00', 41, 94.0),
 ('Daniel Cormier', '2015-10-03 00:00:00', 283, 95.33333333333333),
 ('Daniel Cormier', '2016-07-09 00:00:00', 64, 163.0),
 ('Daniel Cormier', '2017-04-08 00:00:00', 41, 129.33333333333334),
 ('Daniel Cormier', '2017-07-29 00:00:00', 140, 129.33333333333334),
 ('Daniel Cormier', '2018-01-20 00:00:00', 61, 81.66666666666667),
 ('Daniel Cormier', '2018-07-07 00:00:00', 37, 80.66666666666667),
 ('Daniel Cormier', '2018-11-03 00:00:00', 25, 79.33333333333333),
 ('Daniel Cormier', '2019-08-17 00:00:00', 263, 41.0),
 ('Daniel Cormier', '2020-08-15 00:00:00', 183, 108.33333333333333),
 ('Stipe Miocic', '2011-10-08 00:00:00', 142, None),
 ('Stipe 

In [212]:
(187 + 51 + 172) / 3

136.66666666666666

In [213]:

non_fight_stats =  ['fighter_fight_id','fighters','event_date','event_location','matchup','method', 'round','time',
 'time format','referee','"details:"']

fight_stats = [col for col in fighter_stats.columns.to_list() if col not in non_fight_stats]

# window_parts = [f"SUM({c}) OVER w AS sum_{c}" for c in columns]
# sql_query = f"SELECT {', '.join(window_parts)} FROM my_table WINDOW w AS (ORDER BY id);"

test_cols = ['attempted_significant strikes', 'landed_significant strikes', 'attempted_takedowns', 'landed_takedowns']
test_aggs = ['AVG', 'MIN', 'MAX']
only_avg = ['AVG']
test_windows = ['ROWS BETWEEN 1 PRECEDING AND 1 PRECEDING', 'ROWS BETWEEN 2 PRECEDING AND 1 PRECEDING']
windows = {'1fight':'ROWS BETWEEN 1 PRECEDING AND 1 PRECEDING',
                '2fight':'ROWS BETWEEN 2 PRECEDING AND 1 PRECEDING',
                '3fight':'ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING',
                '5fight':'ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING',
                '7fight':'ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING',
                '10fight':'ROWS BETWEEN 10 PRECEDING AND 1 PRECEDING'}

test_queries = []
for col in fight_stats:

    simple_query = '''"{}"'''.format(col)

    test_queries.append(simple_query)

    for time_window in windows:

        for agg in only_avg:

            fighter_agg_query = '''{}("{}") OVER(PARTITION BY fighters \
                ORDER BY event_date ASC \
                    {}) as "{}_{}_{}"'''.format(agg, col, windows[time_window],col,time_window,agg)


            test_queries.append(fighter_agg_query)

    

sql_query = f'''SELECT {', '.join(non_fight_stats)}, \
    {', '.join(test_queries)} \
        FROM fighter_fights \
        WHERE fighters in ("Stipe Miocic", "Daniel Cormier")'''

sql_query

'SELECT fighter_fight_id, fighters, event_date, event_location, matchup, method, round, time, time format, referee, "details:",     "details:", AVG("details:") OVER(PARTITION BY fighters                 ORDER BY event_date ASC                     ROWS BETWEEN 1 PRECEDING AND 1 PRECEDING) as "details:_1fight_AVG", AVG("details:") OVER(PARTITION BY fighters                 ORDER BY event_date ASC                     ROWS BETWEEN 2 PRECEDING AND 1 PRECEDING) as "details:_2fight_AVG", AVG("details:") OVER(PARTITION BY fighters                 ORDER BY event_date ASC                     ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) as "details:_3fight_AVG", AVG("details:") OVER(PARTITION BY fighters                 ORDER BY event_date ASC                     ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) as "details:_5fight_AVG", AVG("details:") OVER(PARTITION BY fighters                 ORDER BY event_date ASC                     ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) as "details:_7fight_AVG", A

In [214]:
len(fight_stats) * 3 * 4

1812

In [215]:
test_query_iterative = cur.execute(sql_query)
query_values = test_query_iterative.fetchall()
query_col_names = test_query_iterative.description

print(query_col_names)
print(query_values)

(('fighter_fight_id', None, None, None, None, None, None), ('fighters', None, None, None, None, None, None), ('event_date', None, None, None, None, None, None), ('event_location', None, None, None, None, None, None), ('matchup', None, None, None, None, None, None), ('method', None, None, None, None, None, None), ('round', None, None, None, None, None, None), ('time', None, None, None, None, None, None), ('format', None, None, None, None, None, None), ('referee', None, None, None, None, None, None), ('details:', None, None, None, None, None, None), ('details:', None, None, None, None, None, None), ('details:_1fight_AVG', None, None, None, None, None, None), ('details:_2fight_AVG', None, None, None, None, None, None), ('details:_3fight_AVG', None, None, None, None, None, None), ('details:_5fight_AVG', None, None, None, None, None, None), ('details:_7fight_AVG', None, None, None, None, None, None), ('details:_10fight_AVG', None, None, None, None, None, None), ('fight_id', None, None, None

In [216]:
test_read_df = pd.read_sql_query(sql_query, conn)
test_read_df

,fighter_fight_id,fighters,event_date,event_location,matchup,method,round,time,format,referee,...,Round 5 landed_ground_5fight_AVG,Round 5 landed_ground_7fight_AVG,Round 5 landed_ground_10fight_AVG,Round 5 attempted_ground,Round 5 attempted_ground_1fight_AVG,Round 5 attempted_ground_2fight_AVG,Round 5 attempted_ground_3fight_AVG,Round 5 attempted_ground_5fight_AVG,Round 5 attempted_ground_7fight_AVG,Round 5 attempted_ground_10fight_AVG
0,932,Daniel Cormier,2013-04-20 00:00:00,"San Jose, California, USA",Frank Mir vs Daniel Cormier,Decision - Unanimous,3,5:00,5:00,Herb Dean,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,15254,Daniel Cormier,2013-10-19 00:00:00,"Houston, Texas, USA",Daniel Cormier vs Roy Nelson,Decision - Unanimous,3,5:00,5:00,Jacob Montalvo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5341,Daniel Cormier,2014-02-22 00:00:00,"Las Vegas, Nevada, USA",Daniel Cormier vs Patrick Cummins,KO/TKO,1,1:19,1:19,Mario Yamasaki,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12507,Daniel Cormier,2014-05-24 00:00:00,"Las Vegas, Nevada, USA",Daniel Cormier vs Dan Henderson,Submission,3,3:53,3:53,Yves Lavigne,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5842,Daniel Cormier,2015-01-03 00:00:00,"Las Vegas, Nevada, USA",Jon Jones vs Daniel Cormier,Decision - Unanimous,5,5:00,5:00,Herb Dean,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
5,12468,Daniel Cormier,2015-05-23 00:00:00,"Las Vegas, Nevada, USA",Anthony Johnson vs Daniel Cormier,Submission,3,2:39,2:39,John McCarthy,...,0.0,0.000000,0.000000,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000
6,14541,Daniel Cormier,2015-10-03 00:00:00,"Houston, Texas, USA",Daniel Cormier vs Alexander Gustafsson,Decision - Split,5,5:00,5:00,Herb Dean,...,0.0,0.000000,0.000000,0.0,NaN,0.0,0.0,0.0,0.000000,0.000000
7,7281,Daniel Cormier,2016-07-09 00:00:00,"Las Vegas, Nevada, USA",Daniel Cormier vs Anderson Silva,Decision - Unanimous,3,5:00,5:00,John McCarthy,...,0.0,0.000000,0.000000,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000
8,289,Daniel Cormier,2017-04-08 00:00:00,"Buffalo, New York, USA",Daniel Cormier vs Anthony Johnson,Submission,2,3:37,3:37,John McCarthy,...,0.0,0.000000,0.000000,NaN,NaN,0.0,0.0,0.0,0.000000,0.000000
9,8360,Daniel Cormier,2017-07-29 00:00:00,"Anaheim, California, USA",Daniel Cormier vs Jon Jones,Overturned,3,3:01,3:01,John McCarthy,...,0.0,0.000000,0.000000,NaN,NaN,NaN,0.0,0.0,0.000000,0.000000


##### Sqlite has a limit on the maximum number of columns, so I cant do the full combination of columns, time periods, and aggregate function. As a result, I will have to do a seperate function call for all 3, then join them together in a single table

In [226]:
non_fight_stats =  ['fight_id','fighter_fight_id','fighters','event_date','event_location','matchup','method', 'round','time',
 'time format','referee','"details:"']

fight_stats = [col for col in fighter_stats.columns.to_list() if col not in non_fight_stats]

test_cols = ['attempted_significant strikes', 'landed_significant strikes', 'attempted_takedowns', 'landed_takedowns']
test_aggs = ['AVG', 'MIN', 'MAX']
only_avg = ['AVG']
test_windows = ['ROWS BETWEEN 1 PRECEDING AND 1 PRECEDING', 'ROWS BETWEEN 2 PRECEDING AND 1 PRECEDING']
windows = {'1fight':'ROWS BETWEEN 1 PRECEDING AND 1 PRECEDING',
                '2fight':'ROWS BETWEEN 2 PRECEDING AND 1 PRECEDING',
                '3fight':'ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING',
                '5fight':'ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING',
                '7fight':'ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING',
                '10fight':'ROWS BETWEEN 10 PRECEDING AND 1 PRECEDING'}
    

def get_agg_queries(columns, time_windows, aggregations):

    partition_queries = []

    for col in columns:

        ## Start by adding the original fight stat column to the list
        simple_query = '''"{}"'''.format(col)
        partition_queries.append(simple_query)

        ## for every combination of time window and aggregation, create a calculated aggregation for the given time window and add it to the list 
        for time_window in time_windows:

            for agg in aggregations:

                fighter_agg_query = '''{}("{}") OVER(PARTITION BY fighters \
                ORDER BY event_date ASC \
                    {}) as "{}_{}_{}"'''.format(agg, col, windows[time_window],col,time_window,agg)
                
                partition_queries.append(fighter_agg_query)

    return partition_queries

test_avg_cols = get_agg_queries(test_cols, windows, ['AVG'])
test_min_cols = get_agg_queries(test_cols, windows, ['MIN'])
test_max_cols = get_agg_queries(test_cols, windows, ['MAX'])


In [227]:
{', '.join(non_fight_stats)}


test_sql_query_avg = f'''SELECT {', '.join(non_fight_stats)}, \
    {', '.join(test_avg_cols)} \
        FROM fighter_fights \
        WHERE fighters in ("Stipe Miocic", "Daniel Cormier")'''

test_sql_query_min = f'''SELECT {', '.join(non_fight_stats)}, \
    {', '.join(test_min_cols)} \
        FROM fighter_fights \
        WHERE fighters in ("Stipe Miocic", "Daniel Cormier")'''

test_sql_query_max = f'''SELECT {', '.join(non_fight_stats)}, \
    {', '.join(test_max_cols)} \
        FROM fighter_fights \
        WHERE fighters in ("Stipe Miocic", "Daniel Cormier")'''

test_sql_query_avg

'SELECT fight_id, fighter_fight_id, fighters, event_date, event_location, matchup, method, round, time, time format, referee, "details:",     "attempted_significant strikes", AVG("attempted_significant strikes") OVER(PARTITION BY fighters                 ORDER BY event_date ASC                     ROWS BETWEEN 1 PRECEDING AND 1 PRECEDING) as "attempted_significant strikes_1fight_AVG", AVG("attempted_significant strikes") OVER(PARTITION BY fighters                 ORDER BY event_date ASC                     ROWS BETWEEN 2 PRECEDING AND 1 PRECEDING) as "attempted_significant strikes_2fight_AVG", AVG("attempted_significant strikes") OVER(PARTITION BY fighters                 ORDER BY event_date ASC                     ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) as "attempted_significant strikes_3fight_AVG", AVG("attempted_significant strikes") OVER(PARTITION BY fighters                 ORDER BY event_date ASC                     ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) as "attempted_signif

In [228]:
test_feat_df_avg = pd.read_sql_query(test_sql_query_avg, conn)
test_feat_df_max = pd.read_sql_query(test_sql_query_max, conn)
test_feat_df_min = pd.read_sql_query(test_sql_query_min, conn)

test_feat_df_avg.head()

,fight_id,fighter_fight_id,fighters,event_date,event_location,matchup,method,round,time,format,...,attempted_takedowns_5fight_AVG,attempted_takedowns_7fight_AVG,attempted_takedowns_10fight_AVG,landed_takedowns,landed_takedowns_1fight_AVG,landed_takedowns_2fight_AVG,landed_takedowns_3fight_AVG,landed_takedowns_5fight_AVG,landed_takedowns_7fight_AVG,landed_takedowns_10fight_AVG
0,76,932,Daniel Cormier,2013-04-20 00:00:00,"San Jose, California, USA",Frank Mir vs Daniel Cormier,Decision - Unanimous,3,5:00,5:00,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,547,15254,Daniel Cormier,2013-10-19 00:00:00,"Houston, Texas, USA",Daniel Cormier vs Roy Nelson,Decision - Unanimous,3,5:00,5:00,...,1.00,1.00,1.00,3,0.0,0.0,0.0,0.0,0.0,0.0
2,1631,5341,Daniel Cormier,2014-02-22 00:00:00,"Las Vegas, Nevada, USA",Daniel Cormier vs Patrick Cummins,KO/TKO,1,1:19,1:19,...,3.00,3.00,3.00,0,3.0,1.5,1.5,1.5,1.5,1.5
3,825,12507,Daniel Cormier,2014-05-24 00:00:00,"Las Vegas, Nevada, USA",Daniel Cormier vs Dan Henderson,Submission,3,3:53,3:53,...,2.00,2.00,2.00,3,0.0,1.5,1.0,1.0,1.0,1.0
4,5877,5842,Daniel Cormier,2015-01-03 00:00:00,"Las Vegas, Nevada, USA",Jon Jones vs Daniel Cormier,Decision - Unanimous,5,5:00,5:00,...,2.25,2.25,2.25,1,3.0,1.5,2.0,1.5,1.5,1.5


In [229]:
def merg_agg_dfs(dfs):

    merged_dfs = dfs[0]

    for i in range(1, len(dfs)):

        to_merge_df = dfs[i]

        merged_dfs = pd.merge(
            left=merged_dfs, 
            right=to_merge_df,
            how='inner',
            left_on=['fighter_fight_id'],
            right_on=['fighter_fight_id']
            )
        
    duplicate_cols = [col for col in merged_dfs if col[-2:] in ['_x', '_y']]

    merged_dfs.drop(columns = duplicate_cols, inplace = True)
    
    return merged_dfs

dfs_test = merg_agg_dfs([test_feat_df_avg, test_feat_df_max, test_feat_df_min])

print(dfs_test.shape)
dfs_test.head()

(34, 88)


,fighter_fight_id,attempted_significant strikes_1fight_AVG,attempted_significant strikes_2fight_AVG,attempted_significant strikes_3fight_AVG,attempted_significant strikes_5fight_AVG,attempted_significant strikes_7fight_AVG,attempted_significant strikes_10fight_AVG,landed_significant strikes_1fight_AVG,landed_significant strikes_2fight_AVG,landed_significant strikes_3fight_AVG,...,attempted_takedowns_5fight_MIN,attempted_takedowns_7fight_MIN,attempted_takedowns_10fight_MIN,landed_takedowns,landed_takedowns_1fight_MIN,landed_takedowns_2fight_MIN,landed_takedowns_3fight_MIN,landed_takedowns_5fight_MIN,landed_takedowns_7fight_MIN,landed_takedowns_10fight_MIN
0,932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,15254,96.0,96.0,96.000000,96.000000,96.000000,96.000000,59.0,59.0,59.000000,...,1.0,1.0,1.0,3,0.0,0.0,0.0,0.0,0.0,0.0
2,5341,148.0,122.0,122.000000,122.000000,122.000000,122.000000,74.0,66.5,66.500000,...,1.0,1.0,1.0,0,3.0,0.0,0.0,0.0,0.0,0.0
3,12507,37.0,92.5,93.666667,93.666667,93.666667,93.666667,18.0,46.0,50.333333,...,0.0,0.0,0.0,3,0.0,0.0,0.0,0.0,0.0,0.0
4,5842,80.0,58.5,88.333333,90.250000,90.250000,90.250000,50.0,34.0,47.333333,...,0.0,0.0,0.0,1,3.0,0.0,0.0,0.0,0.0,0.0


In [230]:
## Create master function that takes main query, parameters for agg columns, and returns a merged and deduplicated dataset

def build_modeling_features(main_query, non_fight_stats, columns, time_windows, aggregations, conn):

    agg_dfs = []

    for agg in aggregations:

        agg_feat_cols = get_agg_queries(columns, time_windows, [agg])

        full_agg_query = main_query.format(', '.join(non_fight_stats),', '.join(agg_feat_cols))

        agg_df = pd.read_sql_query(full_agg_query, conn)

        agg_dfs.append(agg_df)


    merged_df = merg_agg_dfs(agg_dfs)

    return merged_df



In [231]:
test_main_query = '''SELECT {}, \
    {} \
        FROM fighter_fights \
        WHERE fighters in ("Stipe Miocic", "Daniel Cormier")'''


full_main_query = '''SELECT {}, \
    {} \
        FROM fighter_fights'''


test_fight_features_df = build_modeling_features(test_main_query, non_fight_stats, fight_stats, windows, test_aggs, conn)
full_fight_features_df = build_modeling_features(full_main_query, non_fight_stats, fight_stats, windows, test_aggs, conn)



In [232]:
## old shape: (17994, 2833)

print(full_fight_features_df.shape)
full_fight_features_df.tail()

(17208, 2862)


,fighter_fight_id,details:_1fight_AVG,details:_2fight_AVG,details:_3fight_AVG,details:_5fight_AVG,details:_7fight_AVG,details:_10fight_AVG,knockdowns_1fight_AVG,knockdowns_2fight_AVG,knockdowns_3fight_AVG,...,Round 5 landed_ground_5fight_MAX,Round 5 landed_ground_7fight_MAX,Round 5 landed_ground_10fight_MAX,Round 5 attempted_ground,Round 5 attempted_ground_1fight_MAX,Round 5 attempted_ground_2fight_MAX,Round 5 attempted_ground_3fight_MAX,Round 5 attempted_ground_5fight_MAX,Round 5 attempted_ground_7fight_MAX,Round 5 attempted_ground_10fight_MAX
17203,17046,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17204,15746,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17205,4871,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.666667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17206,15436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17207,16063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Now that I have full fighter features, I can join this to my original fight dataframe to get modeling features for each fight row

In [233]:
fights_df.head()

,matchup,method,round,time,time format,referee,details:,fighters_a,fighters_b,knockdowns_a,...,Round 5 attempted_clinch_a,Round 5 landed_clinch_b,Round 5 attempted_clinch_b,Round 5 landed_ground_a,Round 5 attempted_ground_a,Round 5 landed_ground_b,Round 5 attempted_ground_b,event_date,event_location,fight_id
0,Marlon Vera vs Dominick Cruz,KO/TKO,4,2:17,5 Rnd (5-5-5-5-5),Herb Dean,Kick to Head At Distance,Marlon Vera,Dominick Cruz,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-08-13,"San Diego, California, USA",1
1,Nate Landwehr vs David Onama,Decision - Majority,3,5:00,3 Rnd (5-5-5),Mike Beltran,Mike Bell 28 - 28.Junichiro Kamijo 27 - 29.Der...,Nate Landwehr,David Onama,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-08-13,"San Diego, California, USA",2
2,Yazmin Jauregui vs Iasmin Lucindo,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),Jason Herzog,Ron McCarthy 27 - 30.Derek Cleary 28 - 29.Sal ...,Yazmin Jauregui,Iasmin Lucindo,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-08-13,"San Diego, California, USA",3
3,Devin Clark vs Azamat Murzakanov,KO/TKO,3,1:18,3 Rnd (5-5-5),Frank Trigg,Punch to Body At Distance,Devin Clark,Azamat Murzakanov,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-08-13,"San Diego, California, USA",4
4,Priscila Cachoeira vs Ariane da Silva,KO/TKO,1,1:05,3 Rnd (5-5-5),Herb Dean,Punches to Head At Distance,Priscila Cachoeira,Ariane da Silva,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-08-13,"San Diego, California, USA",5


In [ ]:
fights_df['event_date'] = pd.to_datetime(fights_df['event_date'])
full_fight_features_df['event_date'] = pd.to_datetime(full_fight_features_df['event_date'])


fights_fighter_a_feats = pd.merge(
    left=fights_df, 
    right=full_fight_features_df,
    how='left',
    left_on=['fight_id','fighters_a'],
    right_on=['fight_id', 'fighters']
    )
            
fights_full = pd.merge(
            left=fights_fighter_a_feats, 
            right=full_fight_features_df,
            how='left',
            left_on=['fight_id', 'fighters_b'],
            right_on=['fight_id', 'fighters']
            )


In [198]:
print(fights_df.shape)
print(full_fight_features_df.shape)
print(fights_fighter_a_feats.shape)
print(fights_full.shape)


(8604, 309)
(17208, 2861)
(8764, 3169)
(8908, 6029)


In [199]:
counts = full_fight_features_df[['fighters', 'event_date']].value_counts()
counts[counts >= 2]

fighters          event_date
Royce Gracie      1994-03-11    4
Patrick Smith     1994-03-11    4
Gary Goodridge    1996-02-16    3
Marco Ruas        1995-09-08    3
Paul Varelans     1995-09-08    3
Mark Coleman      1996-07-12    3
Dan Severn        1994-12-16    3
Dave Beneteau     1995-04-07    3
Oleg Taktarov     1995-07-14    3
Dan Severn        1995-04-07    3
Oleg Taktarov     1995-12-16    3
Dan Severn        1995-12-16    3
Don Frye          1996-07-12    3
                  1996-12-07    3
                  1996-02-16    3
Remco Pardoel     1994-03-11    3
Royce Gracie      1994-12-16    3
David Abbott      1996-12-07    3
                  1995-07-14    3
Johnny Rhodes     1994-03-11    3
Kenichi Yamamoto  1999-11-19    2
Vitor Belfort     1997-02-07    2
Kevin Jackson     1997-07-27    2
Harold Howard     1994-09-09    2
Dwayne Cason      1997-10-17    2
Kazushi Sakuraba  1997-12-21    2
Jerry Bohlander   1996-02-16    2
Scott Morris      1994-03-11    2
Oleg Taktarov     1

In [168]:
print(fights_fighter_a_feats.shape)
fights_fighter_a_feats.head()

(10546, 3141)


,matchup,method,round,time,time format,referee,details:,fighters_a,fighters_b,knockdowns_a,...,Round 5 landed_ground_5fight_MAX,Round 5 landed_ground_7fight_MAX,Round 5 landed_ground_10fight_MAX,Round 5 attempted_ground,Round 5 attempted_ground_1fight_MAX,Round 5 attempted_ground_2fight_MAX,Round 5 attempted_ground_3fight_MAX,Round 5 attempted_ground_5fight_MAX,Round 5 attempted_ground_7fight_MAX,Round 5 attempted_ground_10fight_MAX
0,Marlon Vera vs Dominick Cruz,KO/TKO,4,2:17,5 Rnd (5-5-5-5-5),Herb Dean,Kick to Head At Distance,Marlon Vera,Dominick Cruz,3,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
1,Nate Landwehr vs David Onama,Decision - Majority,3,5:00,3 Rnd (5-5-5),Mike Beltran,Mike Bell 28 - 28.Junichiro Kamijo 27 - 29.Der...,Nate Landwehr,David Onama,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Yazmin Jauregui vs Iasmin Lucindo,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),Jason Herzog,Ron McCarthy 27 - 30.Derek Cleary 28 - 29.Sal ...,Yazmin Jauregui,Iasmin Lucindo,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Devin Clark vs Azamat Murzakanov,KO/TKO,3,1:18,3 Rnd (5-5-5),Frank Trigg,Punch to Body At Distance,Devin Clark,Azamat Murzakanov,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Priscila Cachoeira vs Ariane da Silva,KO/TKO,1,1:05,3 Rnd (5-5-5),Herb Dean,Punches to Head At Distance,Priscila Cachoeira,Ariane da Silva,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [169]:
print(fights_full.shape)
fights_full.head()

(27316, 5973)


,matchup,method,round,time,time format,referee,details:,fighters_a,fighters_b,knockdowns_a,...,Round 5 landed_ground_5fight_MAX_y,Round 5 landed_ground_7fight_MAX_y,Round 5 landed_ground_10fight_MAX_y,Round 5 attempted_ground_y,Round 5 attempted_ground_1fight_MAX_y,Round 5 attempted_ground_2fight_MAX_y,Round 5 attempted_ground_3fight_MAX_y,Round 5 attempted_ground_5fight_MAX_y,Round 5 attempted_ground_7fight_MAX_y,Round 5 attempted_ground_10fight_MAX_y
0,Marlon Vera vs Dominick Cruz,KO/TKO,4,2:17,5 Rnd (5-5-5-5-5),Herb Dean,Kick to Head At Distance,Marlon Vera,Dominick Cruz,3,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,Nate Landwehr vs David Onama,Decision - Majority,3,5:00,3 Rnd (5-5-5),Mike Beltran,Mike Bell 28 - 28.Junichiro Kamijo 27 - 29.Der...,Nate Landwehr,David Onama,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Yazmin Jauregui vs Iasmin Lucindo,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),Jason Herzog,Ron McCarthy 27 - 30.Derek Cleary 28 - 29.Sal ...,Yazmin Jauregui,Iasmin Lucindo,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Devin Clark vs Azamat Murzakanov,KO/TKO,3,1:18,3 Rnd (5-5-5),Frank Trigg,Punch to Body At Distance,Devin Clark,Azamat Murzakanov,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Priscila Cachoeira vs Ariane da Silva,KO/TKO,1,1:05,3 Rnd (5-5-5),Herb Dean,Punches to Head At Distance,Priscila Cachoeira,Ariane da Silva,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
